In [1]:
import requests
import json
import os

def extraer_resenas_steam(app_id, num_comentarios=100):
    """Extrae reseñas de Steam y las guarda automáticamente en un archivo JSON."""
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    parametros = {
        'language': 'spanish',
        'filter': 'recent',
        'num_per_page': num_comentarios
    }

    try:
        # Realizar petición
        respuesta = requests.get(url, params=parametros)
        respuesta.raise_for_status()
        datos = respuesta.json()

        # Definir ruta de guardado
        nombre_archivo = f"resenas_{app_id}.json"
        ruta_guardado = os.path.join(os.getcwd(), nombre_archivo)

        # Guardar archivo
        with open(ruta_guardado, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=4)

        print(f"✅ Proceso completado. Datos guardados en: {ruta_guardado}")

    except requests.exceptions.RequestException as e:
        print(f"❌ Error de red: {e}")
    except IOError as e:
        print(f"❌ Error al guardar el archivo: {e}")

if __name__ == "__main__":
    # Ejecutar para el ID especificado
    extraer_resenas_steam(4704690, 100)

✅ Proceso completado. Datos guardados en: /content/resenas_4704690.json


In [2]:
import requests
import pandas as pd
import os

def procesar_resenas_steam(app_id, num_comentarios=100):
    """
    Extrae reseñas de Steam, las ordena por valoración y longitud,
    y las exporta a un archivo CSV.
    """
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    params = {
        'language': 'spanish',
        'filter': 'recent',
        'num_per_page': num_comentarios
    }

    try:
        # Petición a la API
        respuesta = requests.get(url, params=params)
        respuesta.raise_for_status()
        data = respuesta.json()

        # Crear DataFrame y limpiar datos
        df = pd.DataFrame(data.get('reviews', []))

        # Extraer campos anidados del autor
        df['usuario_id'] = df['author'].apply(lambda x: x.get('steamid'))
        df['horas_jugadas'] = df['author'].apply(lambda x: x.get('playtime_forever', 0) / 60)

        # Crear indicadores de ordenamiento
        df['es_recomendado'] = df['voted_up'].astype(int)
        df['longitud_mensaje'] = df['review'].apply(len)

        # Ordenar: primero por recomendación (desc), luego por longitud del texto (desc)
        df_final = df.sort_values(
            by=['es_recomendado', 'longitud_mensaje'],
            ascending=[False, False]
        )

        # Seleccionar y renombrar columnas para el CSV final
        columnas = ['usuario_id', 'voted_up', 'horas_jugadas', 'longitud_mensaje', 'review']
        df_export = df_final[columnas].rename(columns={'voted_up': 'recomendado'})

        # Guardar en CSV
        ruta_csv = os.path.join(os.getcwd(), f"resenas_{app_id}_limpio.csv")
        df_export.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

        print(f"✅ Éxito: {len(df_export)} reseñas procesadas.")
        print(f"📂 Archivo guardado en: {ruta_csv}")

    except Exception as e:
        print(f"❌ Error al procesar: {e}")

if __name__ == "__main__":
    procesar_resenas_steam(4704690, 100)

✅ Éxito: 100 reseñas procesadas.
📂 Archivo guardado en: /content/resenas_4704690_limpio.csv


In [3]:
import requests
import pandas as pd
import os
import re

def calcular_indice_ia(texto):
    """
    Asigna una puntuación del 1 al 10 basándose en heurísticas de lenguaje.
    """
    texto = str(texto).lower()
    score = 1

    # Lista de conectores y términos comunes en textos generados por IA
    indicadores = [
        r"\ben conclusión\b", r"\bademás\b", r"\bsin embargo\b",
        r"\bpor otro lado\b", r"\btransformar\b", r"\bexperiencia\b",
        r"\baspectos\b", r"\bfundamental\b", r"\bdetallado\b"
    ]

    # Sumar puntos por vocabulario estructurado
    for patron in indicadores:
        if re.search(patron, texto):
            score += 1

    # Sumar puntos por falta de errores (perfección gramatical sospechosa)
    # Los humanos tienden a usar abreviaciones o cometer errores tipográficos
    if len(texto) > 40 and not re.search(r"[^\w\s.,!¡¿?]", texto):
        score += 2

    return min(score, 10)

def generar_tabla_analisis(app_id, num_comentarios=100):
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    params = {'language': 'spanish', 'filter': 'recent', 'num_per_page': num_comentarios}

    try:
        respuesta = requests.get(url, params=params)
        respuesta.raise_for_status()
        data = respuesta.json()

        # Crear DataFrame
        df = pd.DataFrame(data.get('reviews', []))

        # Limpieza de columnas clave
        df['usuario_id'] = df['author'].apply(lambda x: x.get('steamid'))
        df['horas_jugadas'] = df['author'].apply(lambda x: x.get('playtime_forever', 0) / 60)
        df['voto'] = df['voted_up'].map({True: 'Recomendado', False: 'No recomendado'})

        # --- Aplicar procedimiento de IA ---
        df['indice_ia'] = df['review'].apply(calcular_indice_ia)

        # Ordenar tabla para análisis (mayor índice de IA primero)
        df_final = df[['usuario_id', 'voto', 'horas_jugadas', 'review', 'indice_ia']]
        df_final = df_final.sort_values(by='indice_ia', ascending=False)

        # Guardar como nueva tabla
        ruta_archivo = os.path.join(os.getcwd(), "tabla_analisis_ia.csv")
        df_final.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')

        print(f"✅ Tabla generada con éxito.")
        print(f"📂 Ubicación: {ruta_archivo}")

    except Exception as e:
        print(f"❌ Error al crear la tabla: {e}")

if __name__ == "__main__":
    generar_tabla_analisis(4704690, 100)

✅ Tabla generada con éxito.
📂 Ubicación: /content/tabla_analisis_ia.csv
